# Demonstration 

## Setup

First, it is necessary to import packages. A `Munch` is a `Dict` that acts like a structure, so we can use dot notation. We'll use this to demonstrate different functions and keep the results organized.

In [1]:
import numpy as np
import astropy.units as u
from munch import Munch

import tellurion.core as tell
import tellurion.ork as tork

Next we define a specific epoch time `newyear` and a sequence of time steps every 5 minutes for one hour `prop5m1h`

In [2]:
newyear = tell.abstime('2025-01-01T00:00:00')
prop5m1h = np.linspace(5.0*u.minute, 60.0*u.minute, 12) # Step every 5 minutes for an hour

## State and elements


Create the tree structure `demoa` to hold values; `demoa.init` will hold initial values.

In [3]:
demoa = Munch()
demoa.init = Munch()

Create a position, velocity and time, representing the __[orbital state vector](https://en.wikipedia.org/wiki/Orbital_state_vectors)__

In [4]:
demoa.init.pv = [5740.13268349, 3314.06715, 0., -2.75082684, 4.76457184, 5.50165367]
demoa.init.pvt = tell.pvtcart(demoa.init.pv, newyear)
demoa.init.pvt

PositionVelocityT(time=<Time object: scale='utc' format='isot' value=2025-01-01T00:00:00.000>, aux={})

Transform this into a Kepler element set and display it as a dictionary of dimensioned quantities

In [5]:
demoa.init.kep = tork.kepler(demoa.init.pvt)  # Convert PVT to Kepler elements
demoa.init.kep.els.to_dict()

{'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'ma': <Quantity -0. deg>}

It is often more useful to know the altitudes of perigee and apogee than just the semimajor axis and inclination; this is accomplished with the function `allplane()`, which includes other useful information such as the mean motion

In [6]:
tork.allplane(demoa.init.kep.els.to_dict())

{'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'ma': <Quantity -0. deg>,
 'memo': <Quantity 0.0012 rad / s>,
 'radper': <Quantity 6628.1343 km>,
 'radapo': <Quantity 6716.6145 km>,
 'altper': <Quantity 249.9978 km>,
 'altapo': <Quantity 338.4781 km>}

## Propagation

### Analytical Kepler propagator

Propagate the previous state `demoa.init.pvt` using an analytical two-body propagator and store the computation in `demoa.propa`. The ephemeris table includes the initial state (this is the default but is shown explicitly here) and shows both the time from epoch and the elapsed time from the previous time step.

In [7]:
demoa.propa = Munch()
demoa.propa.gen = tork.prepare(demoa.init.pvt, 1*u.day, forceenv=tork.kepleranalytic())
demoa.propa.ephem = tork.propagate(demoa.propa.gen, prop5m1h, include_init=True)
demoa.propa.ephem

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:00:00.000,0.0s,,5740.133 .. -0.000,-2.750827 .. 5.501654
2025-01-01T00:05:00.000,5min,5min,4581.816 .. 1616.827,-4.891449 .. 5.166423
2025-01-01T00:10:00.000,10min,5min,2865.500 .. 3036.847,-6.432387 .. 4.203822
2025-01-01T00:15:00.000,15min,5min,801.266 .. 4088.442,-7.187958 .. 2.736513
2025-01-01T00:20:00.000,20min,5min,-1359.960 .. 4646.558,-7.074062 .. 0.948419
2025-01-01T00:25:00.000,25min,5min,-3358.333 .. 4647.313,-6.115173 .. -0.941271
2025-01-01T00:30:00.000,30min,5min,-4956.792 .. 4094.285,-4.436283 .. -2.706753
2025-01-01T00:35:00.000,35min,5min,-5968.504 .. 3056.385,-2.243216 .. -4.142427


To get the full vectors, get the `'position'` or `'velocity'` columns explicitly. 

In [8]:
demoa.propa.ephem[3]['position']

<Quantity [ 801.2655, 5183.5367, 4088.4417] km>

Find the altitudes of perigee and apogee; because a two-body propagator is used, they only change due to the slow rotation of the coordinate system.

In [9]:
demoa.propa.altperapo = tork.tselements(demoa.propa.ephem, ["altper","altapo"])
demoa.propa.altperapo

time,altper,altapo
,km,km
Time,float64,float64
2025-01-01T00:00:00.000,249.99783999568226,338.47806177843364
2025-01-01T00:05:00.000,249.99783999568226,338.47806177843734
2025-01-01T00:10:00.000,249.99783999568226,338.47806177843364
2025-01-01T00:15:00.000,249.9978399956841,338.4780617784392
2025-01-01T00:20:00.000,249.99783999568226,338.4780617784299
2025-01-01T00:25:00.000,249.9978399956869,338.47806177843364
2025-01-01T00:30:00.000,249.99783999568132,338.478061778428
2025-01-01T00:35:00.000,249.99783999568783,338.478061778429


### Numerical propagator

Propagation results will be saved in `demoa.prop`.

In [ ]:
demoa.prop = Munch()

Make the propagation generator, which is necessary to create an ephemeris. A maximum time must be specified; we will make it 1 day.

In [ ]:
demoa.prop.gen = tork.prepare(demoa.init.pvt, 1*u.day)

Propagate for 1 hour in 5 minute steps, and include the initial state in the ephemeris table; note the middle (`Y`) value for the position and velocity vectors has been elided, but it is present 

In [ ]:
demoa.prop.ephem = tork.propagate(demoa.prop.gen, prop5m1h, True)
demoa.prop.ephem

## Data at single points

This section shows how to get single-point propagation data. In the first computation, propagate to 12 hours with the previously-defined generator, which is within the maximum allowable time of 1 day for this generator.

Find the state at step 5. The use of `tell.splitsq` in the last step is to make the elements result easier to read; access to components, e.g. `...['raan']`, is the same whether `...` is `demoa.prop.gen.kepler()[0]` or the split form.

In [ ]:
pvt5 = tell.pvt(demoa.prop.ephem[5])
tell.splitsq(pvt5.pv)